In [1]:
from google.colab import drive

drive.mount("/content/gdrive")

Mounted at /content/gdrive


STATE-01 — Initialize

In [2]:
import os
import json
import pandas as pd
import numpy as np

PROJECT_ROOT = "/content/gdrive/MyDrive/CrisisWeave_Project"

STATE_RESULTS_DIR = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Dynamic_State"
)

os.makedirs(STATE_RESULTS_DIR, exist_ok=True)

print("CrisisWeave Dynamic Disaster State initialized.")
print("Output directory:", STATE_RESULTS_DIR)

CrisisWeave Dynamic Disaster State initialized.
Output directory: /content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Dynamic_State


STATE-02 — Load the fused state

In [3]:
fusion_path = os.path.join(
    PROJECT_ROOT,
    "04_Results",
    "Fusion",
    "CrisisWeave_Fused_State_CW001_EV001.json"
)

with open(fusion_path, "r", encoding="utf-8") as f:
    fused_state = json.load(f)

print("Fused state loaded successfully.")
print("Incident:", fused_state["incident_id"])
print("Evidence:", fused_state["evidence_id"])

Fused state loaded successfully.
Incident: CW-001
Evidence: EV-001


STATE-03 — Extract the current situation

In [4]:
fused_severity = fused_state["severity"]["fused_class"]
severity_scores = fused_state["severity"]["scores"]
severity_uncertainty = fused_state["severity"]["uncertainty"]

bert_category = fused_state[
    "humanitarian"
]["bert_category"]

humanitarian_support = fused_state[
    "humanitarian"
]["support_score"]

temporal_trend = fused_state[
    "temporal"
]["trend"]

temporal_escalation = fused_state[
    "temporal"
]["escalation_score"]

progression_direction = fused_state[
    "temporal"
]["progression_direction"]

gis_context_score = fused_state[
    "gis"
]["critical_context_score"]

response_status = fused_state[
    "response"
]["status"]

response_signal = fused_state[
    "response"
]["response_signal"]

overall_conflict = fused_state[
    "conflict"
]["overall_conflict"]

conflict_status = fused_state[
    "conflict"
]["status"]

print("Current fused situation:")
print()
print("Severity:", fused_severity)
print("Humanitarian:", bert_category)
print("Temporal trend:", temporal_trend)
print("Progression:", progression_direction)
print("Response:", response_status)
print("Conflict:", conflict_status)

Current fused situation:

Severity: High
Humanitarian: requests_or_urgent_needs
Temporal trend: INCREASING
Progression: NE
Response: En Route
Conflict: AGREEMENT


STATE-04 — Calculate state confidence

In [5]:
severity_confidence = 1.0 - severity_uncertainty

conflict_factor = {
    "AGREEMENT": 1.0,
    "MINOR_CONFLICT": 0.75,
    "MAJOR_CONFLICT": 0.50
}.get(
    conflict_status,
    0.75
)

state_confidence = (
    0.50 * severity_confidence
    + 0.20 * humanitarian_support
    + 0.15 * temporal_escalation
    + 0.15 * conflict_factor
)

state_confidence = round(
    np.clip(state_confidence, 0, 1),
    4
)

print("Severity confidence:", round(severity_confidence, 4))
print("Conflict factor:", conflict_factor)
print("Overall state confidence:", state_confidence)

Severity confidence: 1.0
Conflict factor: 1.0
Overall state confidence: 1.0


STATE-05 — Determine operational risk level

In [6]:
def determine_risk_level(
    severity,
    temporal_trend,
    humanitarian_support,
    gis_context,
    response_status
):
    score = 0.0

    # Severity
    severity_points = {
        "Low": 1.0,
        "Moderate": 2.0,
        "High": 3.0
    }

    score += severity_points.get(
        severity,
        1.0
    )

    # Escalation
    if temporal_trend == "INCREASING":
        score += 1.0
    elif temporal_trend == "STABLE":
        score += 0.5

    # Humanitarian evidence
    score += humanitarian_support

    # Critical GIS context
    score += gis_context

    # Response status
    if response_status == "Dispatched":
        score += 0.5
    elif response_status == "En Route":
        score += 0.75
    elif response_status == "Arrived":
        score += 1.0

    if score >= 5.5:
        return "CRITICAL"

    if score >= 4.0:
        return "HIGH"

    if score >= 2.5:
        return "MODERATE"

    return "LOW"


risk_level = determine_risk_level(
    fused_severity,
    temporal_trend,
    humanitarian_support,
    gis_context_score,
    response_status
)

print("Operational risk level:", risk_level)

Operational risk level: CRITICAL


STATE-06 — Determine escalation state

In [7]:
if temporal_trend == "INCREASING":
    escalation_state = "ESCALATING"

elif temporal_trend == "DECREASING":
    escalation_state = "DE-ESCALATING"

else:
    escalation_state = "STABLE"


print("Escalation state:", escalation_state)

Escalation state: ESCALATING


STATE-07 — Build the Dynamic Disaster State

In [8]:
dynamic_disaster_state = {
    "incident_id": fused_state["incident_id"],
    "evidence_id": fused_state["evidence_id"],

    "current_state": {
        "severity": fused_severity,
        "severity_scores": severity_scores,
        "operational_risk": risk_level,
        "state_confidence": state_confidence,
        "severity_uncertainty": severity_uncertainty
    },

    "humanitarian_state": {
        "category": bert_category,
        "support_score": humanitarian_support
    },

    "temporal_state": {
        "trend": temporal_trend,
        "escalation_state": escalation_state,
        "escalation_score": temporal_escalation,
        "progression_direction": progression_direction
    },

    "geographic_state": {
        "critical_context_score": gis_context_score
    },

    "response_state": {
        "status": response_status,
        "response_signal": response_signal
    },

    "evidence_quality": {
        "conflict_status": conflict_status,
        "overall_conflict": overall_conflict,
        "state_confidence": state_confidence
    }
}

print(json.dumps(
    dynamic_disaster_state,
    indent=2,
    ensure_ascii=False
))

{
  "incident_id": "CW-001",
  "evidence_id": "EV-001",
  "current_state": {
    "severity": "High",
    "severity_scores": {
      "Low": 0.0,
      "Moderate": 0.0,
      "High": 1.0
    },
    "operational_risk": "CRITICAL",
    "state_confidence": 1.0,
    "severity_uncertainty": -0.0
  },
  "humanitarian_state": {
    "category": "requests_or_urgent_needs",
    "support_score": 1.0
  },
  "temporal_state": {
    "trend": "INCREASING",
    "escalation_state": "ESCALATING",
    "escalation_score": 1.0,
    "progression_direction": "NE"
  },
  "geographic_state": {
    "critical_context_score": 1.0
  },
  "response_state": {
    "status": "En Route",
    "response_signal": 0.75
  },
  "evidence_quality": {
    "conflict_status": "AGREEMENT",
    "overall_conflict": false,
    "state_confidence": 1.0
  }
}


STATE-08 — Create human-readable state summary

In [9]:
state_summary = (
    f"Current disaster severity is {fused_severity}. "
    f"The situation is {escalation_state.lower()} "
    f"with an operational risk level of {risk_level}. "
    f"Humanitarian evidence indicates '{bert_category}'. "
    f"Current spatial progression is toward "
    f"{progression_direction}. "
    f"Response status is '{response_status}'. "
    f"Evidence conflict status is '{conflict_status}'."
)

print(state_summary)

Current disaster severity is High. The situation is escalating with an operational risk level of CRITICAL. Humanitarian evidence indicates 'requests_or_urgent_needs'. Current spatial progression is toward NE. Response status is 'En Route'. Evidence conflict status is 'AGREEMENT'.


STATE-09 — Save Dynamic Disaster State

In [10]:
state_json_path = os.path.join(
    STATE_RESULTS_DIR,
    "CrisisWeave_Dynamic_Disaster_State_CW001.json"
)

with open(
    state_json_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        dynamic_disaster_state,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Dynamic Disaster State saved:")
print(state_json_path)

Dynamic Disaster State saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Dynamic_State/CrisisWeave_Dynamic_Disaster_State_CW001.json


STATE-10 — Save summary table

In [11]:
state_table = pd.DataFrame([{
    "incident_id": dynamic_disaster_state["incident_id"],
    "severity": fused_severity,
    "operational_risk": risk_level,
    "state_confidence": state_confidence,
    "severity_uncertainty": severity_uncertainty,
    "humanitarian_category": bert_category,
    "temporal_trend": temporal_trend,
    "escalation_state": escalation_state,
    "progression_direction": progression_direction,
    "response_status": response_status,
    "conflict_status": conflict_status
}])

state_csv_path = os.path.join(
    STATE_RESULTS_DIR,
    "CrisisWeave_Dynamic_Disaster_State_CW001.csv"
)

state_table.to_csv(
    state_csv_path,
    index=False
)

display(state_table)

print("State table saved:")
print(state_csv_path)

,incident_id,severity,operational_risk,state_confidence,severity_uncertainty,humanitarian_category,temporal_trend,escalation_state,progression_direction,response_status,conflict_status
0,CW-001,High,CRITICAL,1.0,-0.0,requests_or_urgent_needs,INCREASING,ESCALATING,NE,En Route,AGREEMENT


State table saved:
/content/gdrive/MyDrive/CrisisWeave_Project/04_Results/Dynamic_State/CrisisWeave_Dynamic_Disaster_State_CW001.csv
